# 03 · Heat derating of overhead lines (3C): IEEE 738 + NEX-GDDP-CMIP6 for Nagpur

**Question for this notebook:** how much transmission capacity does Nagpur's future heat take away, and how many N-1 contingencies become violations because of it?

**Site:** Nagpur, Maharashtra (21.15°N, 79.09°E, ~310 m)
**Climate data:** NEX-GDDP-CMIP6 (daily, bias-corrected, ~25 km; Thrasher et al., 2022) via the Earth Engine Python API, project `goggle-earth-499404`
**Hourly shape data:** NASA POWER hourly, reused from notebook 02's cache
**Method:** IEEE Std 738-2012 steady-state heat balance, implemented from scratch and validated against the standard's worked example

### Steps
1. Pull daily **Tmax, wind and solar radiation** for every CMIP6 model with all three variables: a **1985–2014 baseline** and a **2041–2070 (≈2050) window**, under **SSP2-4.5 and SSP5-8.5**, for **March–June**.
2. Use NASA POWER hourly data to convert daily-mean radiation into afternoon conditions.
3. Build **hot-day design conditions** per model and period: the 99th-percentile Tmax, with the wind and radiation on those hot days.
4. Compute conductor **ampacity** with IEEE 738 and the **derating ratio** \(I_{2050}/I_{base}\) per model.
5. **Rerun the N-1 analysis of notebook 01** with derated line ratings, with and without heat-driven demand growth (the "double squeeze" of Bartos et al., 2016), and count how many contingencies become violations.

**Run order:** after 01 (needs `network_prepared.json`) and 02 (reuses the NASA POWER cache; 03 downloads it if missing).

**Outputs** (in `results/03_heat/`): `cmip6_nagpur_daily.csv`, `design_conditions.csv`, `ratings_by_model.csv`, `derating_summary.csv`, `derating_ratios.json`, `heat_n1_results.csv`, `heat_n1_summary.csv`, `heat_scenarios_meta.csv`, `heat_bus_shed_timeseries.csv`, figures.

> **About "open source":** the Python libraries and both datasets are open. Earth Engine is free for non-commercial and research use but is a proprietary platform. Section 9 notes a fully open alternative.

## 0. Setup
`earthengine-api` is pre-installed in Colab; the cell below makes sure the rest is available.

In [ ]:
!pip install -q earthengine-api pandapower pvlib

In [ ]:
import os, sys, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})

# ---- Shared project settings (identical in all five notebooks) ----
PROJECT_ID = "goggle-earth-499404"          # Google Cloud project registered for Earth Engine
SITE = "Nagpur, Maharashtra"
LAT, LON, ELEV_M = 21.15, 79.09, 310
TZ = "Asia/Kolkata"

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True                             # keep results in Google Drive so the notebooks can share them
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/grid_stress_nagpur"
else:
    ROOT = os.path.abspath("grid_stress_nagpur")

CACHE = os.path.join(ROOT, "cache")
RES = {k: os.path.join(ROOT, "results", k) for k in
       ["01_contingency", "02_feeder", "03_heat", "04_cascade", "05_mitigation"]}
for d in [CACHE, *RES.values()]:
    os.makedirs(d, exist_ok=True)
OUT = RES["03_heat"]

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, name), dpi=150, bbox_inches="tight")
    plt.show()

def require(path, notebook):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run notebook {notebook} first (with USE_DRIVE = True).")
    return path

print("Project folder:", ROOT)

In [ ]:
# ---- NASA POWER hourly data (shared by notebooks 02, 03 and 05; cached in the project folder) ----
import requests

POWER_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"
POWER_PARAMS = ["ALLSKY_SFC_SW_DWN", "T2M", "WS10M"]

def fetch_power_year(year, lat=LAT, lon=LON, retries=3):
    """One year of hourly GHI (W/m²), 2-m air temperature (°C) and 10-m wind (m/s), in UTC."""
    f = os.path.join(CACHE, f"power_utc_{lat:.2f}_{lon:.2f}_{year}.csv")
    if os.path.exists(f):
        return pd.read_csv(f, index_col=0, parse_dates=True)
    params = {"parameters": ",".join(POWER_PARAMS), "community": "RE",
              "longitude": lon, "latitude": lat, "start": f"{year}0101", "end": f"{year}1231",
              "format": "JSON", "time-standard": "UTC"}
    for k in range(retries):
        try:
            r = requests.get(POWER_URL, params=params, timeout=180)
            r.raise_for_status()
            break
        except Exception:
            if k == retries - 1:
                raise
            time.sleep(10 * (k + 1))
    p = r.json()["properties"]["parameter"]
    df = pd.DataFrame({k: pd.Series(v) for k, v in p.items()})
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    df = df.replace(-999.0, np.nan).sort_index()
    df.columns = [{"ALLSKY_SFC_SW_DWN": "ghi", "T2M": "t2m", "WS10M": "ws10m"}[c] for c in df.columns]
    df.to_csv(f)
    return df

def load_power(years):
    return pd.concat([fetch_power_year(y) for y in years])

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
OUTDIR = OUT
CMIP_CACHE = os.path.join(CACHE, "cmip6")
os.makedirs(CMIP_CACHE, exist_ok=True)

### 0.1 Authenticate Earth Engine
The project `goggle-earth-499404` must be registered for Earth Engine (non-commercial / research use) at <https://code.earthengine.google.com/register>, with the Earth Engine API enabled in that Cloud project. Authentication opens a Google sign-in prompt the first time.

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialised for project", PROJECT_ID)

## 1. Configuration
All study choices live here so they can be quoted directly in the technical note.

In [ ]:
BASELINE = (1985, 2014)       # NEX-GDDP-CMIP6 historical runs end in 2014
FUTURE = (2041, 2070)         # 30-year window centred on 2050
SCENARIOS = ["ssp245", "ssp585"]
MONTHS = (3, 6)               # March–June: pre-monsoon peak heat

PCT_TA = 99                   # design ambient = 99th percentile of daily Tmax in the season
PCT_HOT = 95                  # "hot days" = days with Tmax >= 95th percentile; wind & radiation taken on these days

POWER_YEARS = (2005, 2024)    # NASA POWER hourly years used to derive peak/daily-mean factors (hourly data start 2001)
RAD_FACTOR_MODE = "coincident"   # "coincident" = radiation at the hour of daily Tmax; "peak" = max hourly (more conservative)
APPLY_WIND_FACTOR = False        # True = scale daily-mean CMIP6 wind to the hour of Tmax using NASA POWER
FIXED_WIND = 0.61                # m/s; conventional conservative rating wind (2 ft/s), used as a sensitivity case

GDDP_SCALE_M = 27830          # native 0.25° grid
CHUNK_YEARS = 10              # split each 30-year request into 10-year chunks (keeps EE requests small)
MAX_WORKERS = 8               # parallel Earth Engine requests

## 2. Extract NEX-GDDP-CMIP6 for Nagpur (Earth Engine)
Dataset: `NASA/GDDP-CMIP6` (Thrasher et al., 2022). Each image is one day for one model and one scenario, with the properties `model` and `scenario`.

| Band | Meaning | Unit |
|---|---|---|
| `tasmax` | daily maximum near-surface air temperature | K (converted to °C) |
| `sfcWind` | daily-mean near-surface wind speed | m/s |
| `rsds` | daily-mean surface downwelling shortwave radiation | W/m² |

A few models do not provide every variable; those are filtered out automatically.

In [ ]:
COLL = ee.ImageCollection("NASA/GDDP-CMIP6")
BANDS = ["tasmax", "sfcWind", "rsds"]
POINT = ee.Geometry.Point([LON, LAT])

def models_available(scenario, day):
    ic = (COLL.filterDate(day, ee.Date(day).advance(1, "day"))
              .filter(ee.Filter.eq("scenario", scenario)))
    for b in BANDS:
        ic = ic.filter(ee.Filter.listContains("system:band_names", b))
    return set(ic.aggregate_array("model").getInfo())

model_sets = [models_available("historical", "2000-05-15")]
model_sets += [models_available(s, "2050-05-15") for s in SCENARIOS]
MODELS = sorted(set.intersection(*model_sets))
print(f"{len(MODELS)} models have tasmax, sfcWind and rsds for historical + {SCENARIOS}:")
print(", ".join(MODELS))

In [ ]:
def fetch_chunk(model, scenario, y0, y1, retries=4):
    # Daily values at the Nagpur grid cell for one model, scenario and year range (season only).
    f = os.path.join(CMIP_CACHE, f"{model}_{scenario}_{y0}_{y1}.csv")
    if os.path.exists(f):
        return pd.read_csv(f, parse_dates=["date"])
    ic = (COLL.filter(ee.Filter.eq("model", model))
              .filter(ee.Filter.eq("scenario", scenario))
              .filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
              .filter(ee.Filter.calendarRange(MONTHS[0], MONTHS[1], "month")))
    for b in BANDS:
        ic = ic.filter(ee.Filter.listContains("system:band_names", b))
    ic = ic.select(BANDS)
    for attempt in range(retries):
        try:
            rows = ic.getRegion(POINT, GDDP_SCALE_M).getInfo()
            break
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(5 * (attempt + 1))
    df = pd.DataFrame(rows[1:], columns=rows[0])
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["time"], unit="ms").dt.normalize()
    df = df[["date"] + BANDS].copy()
    df[BANDS] = df[BANDS].astype(float)
    df["tasmax"] -= 273.15
    df = df.groupby("date", as_index=False)[BANDS].mean()   # guards against duplicate images
    df.insert(0, "model", model)
    df.insert(1, "scenario", scenario)
    df.to_csv(f, index=False)
    return df

def year_chunks(y0, y1, n=CHUNK_YEARS):
    return [(a, min(a + n - 1, y1)) for a in range(y0, y1 + 1, n)]

jobs = [(m, "historical", a, b) for m in MODELS for a, b in year_chunks(*BASELINE)]
jobs += [(m, s, a, b) for m in MODELS for s in SCENARIOS for a, b in year_chunks(*FUTURE)]
print(f"{len(jobs)} Earth Engine requests (cached ones are skipped)")

In [ ]:
frames, failed = [], []
t0 = time.time()
with ThreadPoolExecutor(MAX_WORKERS) as pool:
    futs = {pool.submit(fetch_chunk, *j): j for j in jobs}
    for i, fut in enumerate(as_completed(futs), 1):
        try:
            frames.append(fut.result())
        except Exception as e:
            failed.append((futs[fut], str(e)[:120]))
        if i % 25 == 0 or i == len(jobs):
            print(f"  {i}/{len(jobs)} done, {time.time() - t0:.0f}s elapsed")

cmip = pd.concat([f for f in frames if len(f)], ignore_index=True)
cmip.to_csv(os.path.join(OUTDIR, "cmip6_nagpur_daily.csv"), index=False)
print(f"{len(cmip):,} model-days extracted; {len(failed)} failed requests")
for j, e in failed[:10]:
    print("  failed:", j, e)

**If the run is interrupted** (Colab timeout), just re-run the two cells above: finished chunks are cached and skipped.
To reload a finished extraction later without Earth Engine, run the next cell.

In [ ]:
cmip = pd.read_csv(os.path.join(OUTDIR, "cmip6_nagpur_daily.csv"), parse_dates=["date"])
cmip["period"] = np.where(cmip["scenario"] == "historical", "baseline", "future")

# Quality check: keep models with data in every scenario and with enough days
counts = cmip.groupby(["model", "scenario"]).size().unstack()
expected_days = 30 * 122 * 0.95
complete = counts.index[(counts.reindex(columns=["historical"] + SCENARIOS) >= expected_days).all(axis=1)]
dropped = sorted(set(counts.index) - set(complete))
cmip = cmip[cmip["model"].isin(complete)].dropna(subset=["tasmax", "sfcWind", "rsds"])
MODELS = sorted(complete)
print(f"{len(MODELS)} complete models; dropped: {dropped if dropped else 'none'}")
counts.loc[MODELS].head()

## 3. NASA POWER hourly data — from daily means to afternoon peak conditions
CMIP6 gives **daily** values, but a line rating must hold at the **hour of peak conductor heating**: mid-afternoon, when air temperature peaks while the sun is still high.
`rsds` is a 24-hour mean (including night), so it must be scaled up.

From NASA POWER hourly data for the March–June hot days (converted to local solar time, LST = UTC + longitude/15), we compute:
- \(k_{rad}\) = radiation at the hour of daily Tmax ÷ daily-mean radiation (`coincident`), or max hourly ÷ daily mean (`peak`)
- \(k_{wind}\) = wind at the hour of Tmax ÷ daily-mean wind (optional)

These ratios describe the *shape* of a hot day, which is assumed to stay the same in 2050. This is a limitation to state.

In [ ]:
power_utc = load_power(range(POWER_YEARS[0], POWER_YEARS[1] + 1))        # cached by notebook 02
power = power_utc.copy()
power.index = (power_utc.index + pd.Timedelta(hours=LON / 15 + 0.5)).round("h")   # centre of hour, local solar time
power = power[~power.index.duplicated()]
print(power.shape, power.index.min(), "→", power.index.max(), "(LST)")

In [ ]:
def peak_factors(power):
    s = power[(power.index.month >= MONTHS[0]) & (power.index.month <= MONTHS[1])].dropna()
    s = s.assign(day=s.index.normalize(), hour=s.index.hour)
    full = s.groupby("day").size() == 24
    s = s[s["day"].isin(full[full].index)]
    g = s.groupby("day")
    daily = pd.DataFrame({
        "tmax": g["t2m"].max(),
        "hour_tmax": g.apply(lambda d: d.loc[d["t2m"].idxmax(), "hour"], include_groups=False),
        "ghi_mean": g["ghi"].mean(),
        "ghi_max": g["ghi"].max(),
        "ws_mean": g["ws10m"].mean(),
    })
    at_tmax = s.set_index(["day", "hour"])
    idx = list(zip(daily.index, daily["hour_tmax"]))
    daily["ghi_at_tmax"] = at_tmax.loc[idx, "ghi"].values
    daily["ws_at_tmax"] = at_tmax.loc[idx, "ws10m"].values
    hot = daily[daily["tmax"] >= daily["tmax"].quantile(PCT_HOT / 100)]
    out = {
        "n_hot_days": int(len(hot)),
        "typical_hour_of_tmax": int(hot["hour_tmax"].mode().iloc[0]),
        "k_rad_coincident": float((hot["ghi_at_tmax"] / hot["ghi_mean"]).median()),
        "k_rad_peak": float((hot["ghi_max"] / hot["ghi_mean"]).median()),
        "k_wind": float((hot["ws_at_tmax"] / hot["ws_mean"]).median()),
    }
    return out, s, hot

PF, power_season, hot_days_power = peak_factors(power)
K_RAD = PF["k_rad_coincident"] if RAD_FACTOR_MODE == "coincident" else PF["k_rad_peak"]
K_WIND = PF["k_wind"] if APPLY_WIND_FACTOR else 1.0
print(json.dumps(PF, indent=2))
print(f"Using K_RAD = {K_RAD:.2f} ({RAD_FACTOR_MODE}), K_WIND = {K_WIND:.2f}")

In [ ]:
# Diurnal profile on hot days: shows why the afternoon, not the daily mean, sets the rating
hot_hours = power_season[power_season["day"].isin(hot_days_power.index)]
prof = hot_hours.groupby("hour")[["ghi", "t2m", "ws10m"]].median()

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(prof.index, prof["ghi"], color="#d08c00", lw=2, label="GHI (W/m²)")
ax1.axhline(hot_days_power["ghi_mean"].median(), color="#d08c00", ls=":", lw=1.2, label="daily-mean GHI")
ax1.set_xlabel("Hour (local solar time, centre of hour)"); ax1.set_ylabel("GHI (W/m²)")
ax2 = ax1.twinx(); ax2.grid(False)
ax2.plot(prof.index, prof["t2m"], color="#c0392b", lw=2, label="Air temperature (°C)")
ax2.set_ylabel("Air temperature (°C)")
ax1.axvline(PF["typical_hour_of_tmax"], color="grey", ls="--", lw=1)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8, frameon=False)
ax1.set_title(f"{SITE}: median hot-day profile, Mar–Jun (NASA POWER {POWER_YEARS[0]}–{POWER_YEARS[1]})")
savefig("fig_hot_day_profile.png")

## 4. Hot-day design conditions per model and period
For each model and each period (baseline, SSP2-4.5 2050, SSP5-8.5 2050):
- \(T_a\) = 99th percentile of daily Tmax over March–June (the design ambient temperature)
- hot days = days with Tmax ≥ 95th percentile; on those days take the **median** wind and radiation (so wind and sun are *coincident* with the heat, not climatological averages)
- \(Q_{solar} = k_{rad} \times rsds\) and \(V_w = k_{wind} \times sfcWind\)

Working **per model** and then taking the ratio cancels much of each model's own bias.

In [ ]:
def design_conditions(d):
    thr = d["tasmax"].quantile(PCT_HOT / 100)
    hot = d[d["tasmax"] >= thr]
    return pd.Series({
        "Ta": d["tasmax"].quantile(PCT_TA / 100),
        "Tmax_mean": d["tasmax"].mean(),
        "wind_daily": hot["sfcWind"].median(),
        "rsds_daily": hot["rsds"].median(),
        "n_days": len(d), "n_hot": len(hot),
    })

dc = cmip.groupby(["model", "scenario"]).apply(design_conditions, include_groups=False).reset_index()
dc["wind"] = dc["wind_daily"] * K_WIND
dc["Q_solar"] = dc["rsds_daily"] * K_RAD
dc.to_csv(os.path.join(OUTDIR, "design_conditions.csv"), index=False)

summary_dc = (dc.groupby("scenario")[["Ta", "Tmax_mean", "wind", "Q_solar"]]
                .agg(["median", lambda x: x.quantile(.1), lambda x: x.quantile(.9)]))
summary_dc.columns = [f"{a}_{'p50' if b == 'median' else ('p10' if b == '<lambda_0>' else 'p90')}" for a, b in summary_dc.columns]
summary_dc

In [ ]:
# Change in design ambient temperature, per model (future minus baseline)
base = dc[dc["scenario"] == "historical"].set_index("model")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, var, lab in zip(axes, ["Ta", "wind", "Q_solar"],
                        ["Δ design Tmax (°C)", "Δ hot-day wind (m/s)", "Δ afternoon radiation (W/m²)"]):
    data = [(dc[dc["scenario"] == s].set_index("model")[var] - base[var]).dropna() for s in SCENARIOS]
    ax.boxplot(data, tick_labels=[s.upper() for s in SCENARIOS], widths=0.5)
    for i, d in enumerate(data, 1):
        ax.scatter(np.random.normal(i, 0.05, len(d)), d, s=10, alpha=0.6, color="#2c6fbb")
    ax.axhline(0, color="k", lw=0.8); ax.set_title(lab, fontsize=10)
fig.suptitle(f"{SITE}: change in hot-day conditions, {FUTURE[0]}–{FUTURE[1]} vs {BASELINE[0]}–{BASELINE[1]} (each dot = one CMIP6 model)", fontsize=10)
savefig("fig_delta_conditions.png")

## 5. IEEE Std 738-2012 steady-state heat balance (implemented from scratch)

At the maximum allowable conductor temperature \(T_s\), heat gained equals heat lost:

$$q_c + q_r = q_s + I^2 R(T_s) \quad\Rightarrow\quad I = \sqrt{\frac{q_c + q_r - q_s}{R(T_s)}}$$

All terms are per metre of conductor (W/m). SI forms:

**Air properties** at film temperature \(T_{film} = (T_s + T_a)/2\):
- dynamic viscosity \(\mu_f = \dfrac{1.458\times10^{-6}(T_{film}+273)^{1.5}}{T_{film}+383.4}\)
- density \(\rho_f = \dfrac{1.293 - 1.525\times10^{-4}H_e + 6.379\times10^{-9}H_e^2}{1 + 0.00367\,T_{film}}\) (elevation \(H_e\) in m)
- thermal conductivity \(k_f = 2.424\times10^{-2} + 7.477\times10^{-5}T_{film} - 4.407\times10^{-9}T_{film}^2\)

**Convective cooling** \(q_c\) — the dominant term. Reynolds number \(N_{Re} = D\rho_f V_w/\mu_f\); wind direction factor \(K_{angle} = 1.194 - \cos\phi + 0.194\cos 2\phi + 0.368\sin 2\phi\) (= 1 for wind perpendicular to the line):
- \(q_{c1} = K_{angle}\,[1.01 + 1.35 N_{Re}^{0.52}]\,k_f\,(T_s - T_a)\) (low wind)
- \(q_{c2} = K_{angle}\,0.754 N_{Re}^{0.6}\,k_f\,(T_s - T_a)\) (high wind)
- \(q_{cn} = 3.645\,\rho_f^{0.5} D^{0.75} (T_s - T_a)^{1.25}\) (natural convection, still air)
- \(q_c = \max(q_{c1}, q_{c2}, q_{cn})\)

**Radiative cooling:** \(q_r = 17.8\,D\,\varepsilon\left[\left(\tfrac{T_s+273}{100}\right)^4 - \left(\tfrac{T_a+273}{100}\right)^4\right]\) (17.8 = Stefan–Boltzmann constant × π × 10⁸)

**Solar heating:** \(q_s = \alpha\,Q_{se}\sin\theta\,D\). The standard computes \(Q_{se}\sin\theta\) from sun position (clear-sky polynomial, implemented below for validation). For the climate analysis we use the climate-data radiation \(Q_{solar}\) as the flux on the conductor — a stated simplification.

**Resistance** at \(T_s\): linear interpolation between two tabulated AC resistances, \(R(T_s) = R_{lo} + \dfrac{R_{hi}-R_{lo}}{T_{hi}-T_{lo}}(T_s - T_{lo})\).

**Why heat lowers the rating:** higher \(T_a\) shrinks \((T_s - T_a)\), which cuts both \(q_c\) and \(q_r\); more sun raises \(q_s\); weaker wind cuts \(q_c\).

In [ ]:
def air_properties(T_film, elev_m):
    mu = 1.458e-6 * (T_film + 273.0) ** 1.5 / (T_film + 383.4)                              # Pa·s
    rho = (1.293 - 1.525e-4 * elev_m + 6.379e-9 * elev_m ** 2) / (1 + 0.00367 * T_film)      # kg/m³
    k = 2.424e-2 + 7.477e-5 * T_film - 4.407e-9 * T_film ** 2                                # W/(m·°C)
    return mu, rho, k

def convective_cooling(Ts, Ta, D, wind, elev_m, phi_deg=90.0):
    T_film = (Ts + Ta) / 2.0
    mu, rho, k = air_properties(T_film, elev_m)
    phi = np.radians(phi_deg)
    K_angle = 1.194 - np.cos(phi) + 0.194 * np.cos(2 * phi) + 0.368 * np.sin(2 * phi)
    N_re = D * rho * wind / mu
    qc1 = K_angle * (1.01 + 1.35 * N_re ** 0.52) * k * (Ts - Ta)
    qc2 = K_angle * 0.754 * N_re ** 0.6 * k * (Ts - Ta)
    qcn = 3.645 * rho ** 0.5 * D ** 0.75 * np.maximum(Ts - Ta, 0) ** 1.25
    return np.maximum(np.maximum(qc1, qc2), qcn)

def radiative_cooling(Ts, Ta, D, emissivity):
    return 17.8 * D * emissivity * (((Ts + 273.0) / 100) ** 4 - ((Ta + 273.0) / 100) ** 4)

def solar_heating(Q, D, absorptivity):
    return absorptivity * Q * D

def clear_sky_flux(lat_deg, day_of_year, hour, line_azimuth_deg, elev_m, atmosphere="clear"):
    # IEEE 738-2012 clear-sky solar flux on the conductor, Qse·sin(theta) in W/m²
    lat = np.radians(lat_deg)
    decl = np.radians(23.46 * np.sin(np.radians((284 + day_of_year) / 365.0 * 360.0)))
    omega = np.radians((hour - 12.0) * 15.0)
    Hc = np.arcsin(np.cos(lat) * np.cos(decl) * np.cos(omega) + np.sin(lat) * np.sin(decl))
    chi = np.sin(omega) / (np.sin(lat) * np.cos(omega) - np.cos(lat) * np.tan(decl))
    if np.degrees(omega) < 0:
        C = 0.0 if chi >= 0 else 180.0
    else:
        C = 180.0 if chi >= 0 else 360.0
    Zc = C + np.degrees(np.arctan(chi))
    coeffs = {"clear": [-42.2391, 63.8044, -1.9220, 3.46921e-2, -3.61118e-4, 1.94318e-6, -4.07608e-9],
              "industrial": [53.1821, 14.2110, 6.6138e-1, -3.1658e-2, 5.4654e-4, -4.3446e-6, 1.3236e-8]}[atmosphere]
    Hd = np.degrees(Hc)
    Qs = sum(c * Hd ** i for i, c in enumerate(coeffs))
    K_solar = 1 + 1.148e-4 * elev_m - 1.108e-8 * elev_m ** 2
    theta = np.arccos(np.cos(Hc) * np.cos(np.radians(Zc - line_azimuth_deg)))
    return K_solar * Qs * np.sin(theta), Hd, Zc

def ac_resistance(Ts, c):
    return c["R_lo"] + (c["R_hi"] - c["R_lo"]) / (c["T_hi"] - c["T_lo"]) * (Ts - c["T_lo"])

def ampacity(Ta, wind, Q_solar, c, Ts=None, elev_m=0.0, phi_deg=90.0, terms=False):
    Ts = c["Ts_max"] if Ts is None else Ts
    qc = convective_cooling(Ts, Ta, c["D"], wind, elev_m, phi_deg)
    qr = radiative_cooling(Ts, Ta, c["D"], c["emissivity"])
    qs = solar_heating(Q_solar, c["D"], c["absorptivity"])
    R = ac_resistance(Ts, c)
    I = np.sqrt(np.maximum(qc + qr - qs, 0) / R)
    return (I, dict(qc=qc, qr=qr, qs=qs, R=R)) if terms else I

### 5.1 Validation against the IEEE 738-2012 worked example
Drake 26/7 ACSR, \(T_s\) = 100 °C, \(T_a\) = 40 °C, wind 0.61 m/s perpendicular, ε = α = 0.8, latitude 30°N, 10 June, 11:00, east–west line, clear atmosphere, sea level.
The standard's Annex reports a rating of about **1,025 A** (\(q_c\) ≈ 81.9 W/m, \(q_r\) ≈ 39.1 W/m, \(q_s\) ≈ 22.4 W/m).

In [ ]:
DRAKE = dict(name="Drake 26/7 ACSR", D=0.02814, R_lo=7.283e-5, T_lo=25.0, R_hi=8.688e-5, T_hi=75.0,
             emissivity=0.8, absorptivity=0.8, Ts_max=100.0)

Q_ex, Hc_ex, Zc_ex = clear_sky_flux(30, 161, 11, 90, 0, "clear")
I_ex, t = ampacity(40, 0.61, Q_ex, DRAKE, elev_m=0, terms=True)
print(f"Sun altitude {Hc_ex:.1f}°, azimuth {Zc_ex:.1f}°, flux on conductor {Q_ex:.0f} W/m²")
print(f"qc = {t['qc']:.1f} W/m, qr = {t['qr']:.1f} W/m, qs = {t['qs']:.1f} W/m, R(100°C) = {t['R']*1e5:.3f}e-5 ohm/m")
print(f"Rating = {I_ex:.0f} A  (IEEE 738-2012 example ≈ 1025 A)")
assert abs(I_ex - 1025) < 10, "Implementation does not reproduce the standard's example"

### 5.2 Conductor used for the Nagpur analysis
The default is **ACSR Zebra (54/7)**, common on Indian 220 kV and 400 kV lines. The resistances below are **approximate** (DC resistance at 20 °C from IS 398 Part 2, corrected for temperature and a ~2% AC skin effect). Replace them with a manufacturer datasheet for final results.

\(T_{s,max}\) = 75 °C is the traditional Indian ACSR design limit; many utilities now allow 85 °C, so run both.

In [ ]:
R20_DC = 0.06868e-3      # ohm/m, ACSR Zebra DC resistance at 20 °C (IS 398-2)
ALPHA_AL = 0.00403       # 1/°C temperature coefficient of aluminium
SKIN = 1.02              # approximate AC/DC ratio
r_at = lambda T: R20_DC * (1 + ALPHA_AL * (T - 20)) * SKIN

ZEBRA = dict(name="ACSR Zebra 54/7", D=0.02862, R_lo=r_at(25), T_lo=25.0, R_hi=r_at(75), T_hi=75.0,
             emissivity=0.8, absorptivity=0.8, Ts_max=75.0)
COND = ZEBRA
TS_CASES = [75.0, 85.0]
print({k: (f"{v:.3e}" if isinstance(v, float) and v < 1e-2 else v) for k, v in COND.items()})

## 6. Ratings and derating ratios
For each model: \(I_{base}\) from the historical design conditions and \(I_{2050}\) from each scenario; then

$$\text{derating ratio} = I_{2050} / I_{base}$$

Two wind cases:
- **CMIP6 wind** — hot-day wind from each model and period (captures projected changes in wind too)
- **Fixed 0.61 m/s** — same conservative wind in both periods, so the ratio reflects temperature and radiation only. This is the more defensible headline number, because downscaled wind is the least reliable input.

In [ ]:
rows = []
for Ts in TS_CASES:
    for wind_case in ["CMIP6 wind", f"fixed {FIXED_WIND} m/s"]:
        for _, r in dc.iterrows():
            w = r["wind"] if wind_case == "CMIP6 wind" else FIXED_WIND
            rows.append(dict(model=r["model"], scenario=r["scenario"], Ts_max=Ts, wind_case=wind_case,
                             Ta=r["Ta"], wind=w, Q_solar=r["Q_solar"],
                             I=float(ampacity(r["Ta"], w, r["Q_solar"], COND, Ts=Ts, elev_m=ELEV_M))))
rat = pd.DataFrame(rows)

base_I = (rat[rat["scenario"] == "historical"]
          .set_index(["model", "Ts_max", "wind_case"])["I"].rename("I_base"))
fut = rat[rat["scenario"] != "historical"].join(base_I, on=["model", "Ts_max", "wind_case"])
fut["ratio"] = fut["I"] / fut["I_base"]
fut.to_csv(os.path.join(OUTDIR, "ratings_by_model.csv"), index=False)

summ = (fut.groupby(["Ts_max", "wind_case", "scenario"])
           .agg(I_base_p50=("I_base", "median"), I_2050_p50=("I", "median"),
                ratio_p50=("ratio", "median"),
                ratio_p10=("ratio", lambda x: x.quantile(.1)),
                ratio_p90=("ratio", lambda x: x.quantile(.9)),
                ratio_min=("ratio", "min"), n_models=("ratio", "size"))
           .reset_index())
summ["capacity_loss_%_p50"] = (1 - summ["ratio_p50"]) * 100
summ.to_csv(os.path.join(OUTDIR, "derating_summary.csv"), index=False)
summ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, wc in zip(axes, ["CMIP6 wind", f"fixed {FIXED_WIND} m/s"]):
    sub = fut[(fut["wind_case"] == wc) & (fut["Ts_max"] == COND["Ts_max"])]
    data = [sub[sub["scenario"] == s]["ratio"] for s in SCENARIOS]
    ax.boxplot(data, tick_labels=[s.upper() for s in SCENARIOS], widths=0.5)
    for i, d in enumerate(data, 1):
        ax.scatter(np.random.normal(i, 0.05, len(d)), d, s=12, alpha=0.6, color="#2c6fbb")
    ax.axhline(1, color="k", lw=0.8); ax.set_title(f"{wc}, Ts,max = {COND['Ts_max']:.0f} °C", fontsize=10)
axes[0].set_ylabel("Derating ratio  I(2050) / I(baseline)")
fig.suptitle(f"{SITE}: {COND['name']} ampacity derating, {FUTURE[0]}–{FUTURE[1]} (each dot = one CMIP6 model)", fontsize=10)
savefig("fig_derating_ratio.png")

In [ ]:
# Sensitivity: how rating responds to each driver (others held at the baseline ensemble median)
b = dc[dc["scenario"] == "historical"][["Ta", "wind", "Q_solar"]].median()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
Ta_r = np.linspace(38, 52, 50); W_r = np.linspace(0.3, 5, 50); Q_r = np.linspace(400, 1100, 50)
for Ts, ls in zip(TS_CASES, ["-", "--"]):
    axes[0].plot(Ta_r, ampacity(Ta_r, FIXED_WIND, b["Q_solar"], COND, Ts=Ts, elev_m=ELEV_M), ls, label=f"Ts,max {Ts:.0f} °C")
    axes[1].plot(W_r, ampacity(b["Ta"], W_r, b["Q_solar"], COND, Ts=Ts, elev_m=ELEV_M), ls)
    axes[2].plot(Q_r, ampacity(b["Ta"], FIXED_WIND, Q_r, COND, Ts=Ts, elev_m=ELEV_M), ls)
axes[0].set_xlabel("Ambient temperature (°C)"); axes[1].set_xlabel("Wind speed (m/s)"); axes[2].set_xlabel("Solar flux (W/m²)")
axes[0].set_ylabel("Ampacity (A)"); axes[0].legend(frameon=False, fontsize=8)
for ax in axes: ax.set_ylim(bottom=0)
fig.suptitle(f"{COND['name']}: sensitivity of the IEEE 738 rating", fontsize=10)
savefig("fig_sensitivity.png")

## 7. Rerun the N-1 analysis with derated line ratings
The derating ratio multiplies the thermal rating of every **overhead line** (`max_i_ka`) in the network prepared by notebook 01. Transformers are left unchanged, since their ratings follow different standards (IEC 60076-7).

Cases:
- **baseline**: ratings as in notebook 01
- **median model**: ensemble-median derating ratio for each SSP
- **hot model (p10)**: 10th-percentile ratio (a hotter-than-median model), as the stress case
- **median + demand**: derating *and* heat-driven peak-demand growth of `LOAD_SENS_PCT_PER_C` per °C of design-temperature rise. This is the "double squeeze" of Bartos et al. (2016). Indian cooling-driven demand sensitivities of roughly 2–4% per °C are reported in the literature; treat the value as an assumption and vary it.

The headline wind case is the fixed 0.61 m/s, because downscaled wind is the least reliable input.

In [ ]:
# ---- Core grid functions (shared by notebooks 01 and 03) ----
import copy
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import pandapower.topology as top
import scipy.sparse as sp
from scipy.optimize import linprog

SQRT3 = np.sqrt(3.0)


def branch_flows_mva(net):
    """Apparent power through each line and transformer (max of both ends), MVA."""
    rl = net.res_line
    s_line = np.maximum(np.hypot(rl.p_from_mw, rl.q_from_mvar), np.hypot(rl.p_to_mw, rl.q_to_mvar))
    if len(net.trafo):
        rt = net.res_trafo
        s_tr = np.maximum(np.hypot(rt.p_hv_mw, rt.q_hv_mvar), np.hypot(rt.p_lv_mw, rt.q_lv_mvar))
    else:
        s_tr = pd.Series(dtype=float)
    return s_line, s_tr


def assign_ratings(net, mode="base_flow", target_loading=0.7, floor_mva=None):
    """Give the test network realistic thermal ratings.

    mode="case": keep the ratings that come with the test case.
    mode="base_flow": rating = max(|S_base| / target_loading, voltage-level floor).
      This makes the base case secure (every branch <= target_loading) while leaving
      limited headroom, a common way to rate IEEE cases that ship without limits.
    Transformer impedances are kept constant when their MVA rating is changed.
    """
    if mode == "case":
        return net
    floor_mva = floor_mva or {}
    pp.runpp(net)
    s_line, s_tr = branch_flows_mva(net)
    vn_line = net.bus.loc[net.line.from_bus, "vn_kv"].values
    floor_l = np.array([floor_mva.get(v, 0.0) for v in vn_line])
    rating_l = np.maximum(s_line.values / target_loading, floor_l)
    rating_l = np.ceil(rating_l / 5) * 5                                 # round up to 5 MVA steps
    net.line["max_i_ka"] = rating_l / (SQRT3 * vn_line * net.line.parallel.values)
    if len(net.trafo):
        vn_tr = net.bus.loc[net.trafo.hv_bus, "vn_kv"].values
        floor_t = np.array([floor_mva.get(v, 0.0) for v in vn_tr])
        rating_t = np.ceil(np.maximum(s_tr.values / target_loading, floor_t) / 5) * 5
        k = rating_t / net.trafo.sn_mva.values                           # keep ohmic impedance unchanged
        net.trafo["vk_percent"] *= k
        net.trafo["vkr_percent"] *= k
        net.trafo["sn_mva"] = rating_t
    pp.runpp(net)
    return net


def branch_table(net):
    """All lines and transformers with DC-model reactance (p.u. on net.sn_mva) and MVA rating."""
    base = net.sn_mva
    L = net.line
    vn = net.bus.loc[L.from_bus, "vn_kv"].values
    zb = vn ** 2 / base
    lines = pd.DataFrame({
        "kind": "line", "idx": L.index.values,
        "f": L.from_bus.values, "t": L.to_bus.values,
        "x_pu": L.x_ohm_per_km.values * L.length_km.values / L.parallel.values / zb,
        "rating_mva": SQRT3 * vn * L.max_i_ka.values * L.parallel.values * L.df.values,
        "in_service": L.in_service.values,
    })
    tabs = [lines]
    if len(net.trafo):
        T = net.trafo
        vk = np.sqrt(np.maximum(T.vk_percent.values ** 2 - T.vkr_percent.values ** 2, 1e-9)) / 100
        tabs.append(pd.DataFrame({
            "kind": "trafo", "idx": T.index.values,
            "f": T.hv_bus.values, "t": T.lv_bus.values,
            "x_pu": vk * base / T.sn_mva.values / T.parallel.values,
            "rating_mva": T.sn_mva.values * T.parallel.values,
            "in_service": T.in_service.values,
        }))
    bt = pd.concat(tabs, ignore_index=True)
    bt["name"] = bt["kind"].str[0].str.upper() + bt["idx"].astype(str) + " (" + bt["f"].astype(str) + "-" + bt["t"].astype(str) + ")"
    return bt


def bus_loads(net):
    """Net active demand per bus (loads minus static generators), MW."""
    d = net.load[net.load.in_service].groupby("bus")["p_mw"].sum() if len(net.load) else pd.Series(dtype=float)
    if len(net.sgen):
        d = d.sub(net.sgen[net.sgen.in_service].groupby("bus")["p_mw"].sum(), fill_value=0)
    return d.reindex(net.bus.index, fill_value=0.0)


def generator_table(net):
    g = net.gen[net.gen.in_service]
    e = net.ext_grid[net.ext_grid.in_service]
    cost = {}
    if len(net.poly_cost):
        for _, r in net.poly_cost.iterrows():
            cost[(r.et, r.element)] = r.cp1_eur_per_mw
    gens = pd.DataFrame({
        "et": ["gen"] * len(g) + ["ext_grid"] * len(e),
        "element": list(g.index) + list(e.index),
        "bus": list(g.bus) + list(e.bus),
        "p0": list(g.p_mw) + [np.nan] * len(e),
        "pmax": list(g.max_p_mw) + list(e.max_p_mw if "max_p_mw" in e else [1e4] * len(e)),
    })
    gens["pmax"] = gens["pmax"].fillna(1e4)
    gens["cost"] = [cost.get((a, b), 20.0) for a, b in zip(gens.et, gens.element)]
    return gens


class DCLoadShed:
    """DC optimal power flow that treats every load as sheddable.

    minimise   VOLL * sum(shed) + sum(c_g * P_g)
    subject to B*theta = Cg*P_g - (P_d - shed)      (nodal balance, per unit)
               |b_l (theta_f - theta_t)| <= rating_l   (thermal limits)
               0 <= P_g <= P_g,max ; 0 <= shed <= P_d
               theta = 0 at one bus in every island
    Islands with no generation are forced to shed all their load.
    """

    def __init__(self, net, voll=10000.0, redispatch_frac=None, rating_scale=None):
        self.net = net
        self.base = net.sn_mva
        self.bt = branch_table(net)
        if rating_scale is not None:                      # e.g. heat derating of lines only
            self.bt.loc[self.bt.kind == "line", "rating_mva"] *= rating_scale
        self.buses = net.bus.index[net.bus.in_service].values
        self.pos = {b: i for i, b in enumerate(self.buses)}
        self.pd = bus_loads(net).reindex(self.buses).clip(lower=0).values
        self.gens = generator_table(net)
        self.voll = voll
        self.redispatch_frac = redispatch_frac
        self.key = {(k, i): n for n, (k, i) in enumerate(zip(self.bt.kind, self.bt.idx))}

    def solve(self, outages=(), return_flows=False):
        bt = self.bt
        on = bt.in_service.values.copy()
        for o in outages:
            on[self.key[o]] = False
        br = bt[on]
        n, m, ng = len(self.buses), len(br), len(self.gens)
        f = np.array([self.pos[b] for b in br.f]); t = np.array([self.pos[b] for b in br.t])
        b = 1.0 / br.x_pu.values
        C = sp.csr_matrix((np.r_[np.ones(m), -np.ones(m)], (np.r_[np.arange(m), np.arange(m)], np.r_[f, t])), shape=(m, n))
        Bf = sp.diags(b) @ C                                   # branch flow = Bf @ theta  (p.u.)
        Bbus = C.T @ Bf
        gb = np.array([self.pos[x] for x in self.gens.bus])
        Cg = sp.csr_matrix((np.ones(ng), (gb, np.arange(ng))), shape=(n, ng))
        # islands -> one reference angle each
        import networkx as nx
        G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(zip(f, t))
        islands = list(nx.connected_components(G))
        refs = [min(c) for c in islands]
        # variables: theta (n) | Pg (ng) | shed (n)
        nv = n + ng + n
        A_eq = sp.hstack([Bbus, -Cg, -sp.eye(n)]).tocsr()
        b_eq = -self.pd / self.base
        R = sp.csr_matrix((np.ones(len(refs)), (np.arange(len(refs)), refs)), shape=(len(refs), nv))
        A_eq = sp.vstack([A_eq, R]).tocsr(); b_eq = np.r_[b_eq, np.zeros(len(refs))]
        rate = br.rating_mva.values / self.base
        A_ub = sp.vstack([sp.hstack([Bf, sp.csr_matrix((m, ng + n))]),
                          sp.hstack([-Bf, sp.csr_matrix((m, ng + n))])]).tocsr()
        b_ub = np.r_[rate, rate]
        pmax = self.gens.pmax.values.copy()
        if self.redispatch_frac is not None:                  # limited corrective re-dispatch
            p0 = self.gens.p0.values
            up = np.where(np.isnan(p0), pmax, np.minimum(pmax, p0 + self.redispatch_frac * pmax))
            pmax = up
        bounds = [(None, None)] * n + [(0, p / self.base) for p in pmax] + [(0, d / self.base) for d in self.pd]
        cost = np.r_[np.zeros(n), self.gens.cost.values / 1000.0, np.full(n, self.voll / 1000.0)]
        res = linprog(cost, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
        if res.status != 0:
            return dict(ok=False, shed_mw=np.nan, shed_bus=None, n_islands=len(islands))
        x = res.x
        shed = x[n + ng:] * self.base
        out = dict(ok=True, shed_mw=float(shed.sum()), shed_bus=pd.Series(shed, index=self.buses),
                   n_islands=len(islands), pg=x[n:n + ng] * self.base)
        if return_flows:
            out["flows"] = pd.Series(Bf @ x[:n] * self.base, index=br.index)
        return out


def set_status(net, outages, status):
    for kind, idx in outages:
        net[kind].at[idx, "in_service"] = status


def ac_contingency(net, outages, pi_n=1):
    """AC power flow with the given (kind, idx) elements out of service. Restores the network afterwards."""
    set_status(net, outages, False)
    r = dict(converged=False)
    try:
        unsup = top.unsupplied_buses(net)
        r["n_isolated_buses"] = len(unsup)
        r["ac_load_isolated_mw"] = float(net.load[net.load.bus.isin(unsup) & net.load.in_service].p_mw.sum())
        try:
            pp.runpp(net, init="dc")
        except pp.LoadflowNotConverged:
            pp.runpp(net, init="flat", max_iteration=50)
        r["converged"] = True
        ll = net.res_line.loading_percent.where(net.line.in_service).dropna()
        tl = net.res_trafo.loading_percent.where(net.trafo.in_service).dropna() if len(net.trafo) else pd.Series(dtype=float)
        load = pd.concat([ll, tl])
        r["max_loading_pct"] = float(load.max())
        r["n_overloads"] = int((load > 100).sum())
        r["overload_excess_pct"] = float((load - 100).clip(lower=0).sum())
        r["overloaded"] = ";".join([f"L{i}" for i in ll[ll > 100].index] + [f"T{i}" for i in tl[tl > 100].index])
        r["PI"] = float(((load / 100) ** (2 * pi_n)).sum() / (2 * pi_n))       # Ejebe & Wollenberg (1979)
        vm = net.res_bus.vm_pu.dropna()
        vmin_lim = net.bus.loc[vm.index, "min_vm_pu"].fillna(0.95)
        vmax_lim = net.bus.loc[vm.index, "max_vm_pu"].fillna(1.05)
        r["vmin_pu"], r["vmax_pu"] = float(vm.min()), float(vm.max())
        r["n_voltage_violations"] = int(((vm < vmin_lim) | (vm > vmax_lim)).sum())
    except pp.LoadflowNotConverged:
        pass
    finally:
        set_status(net, outages, True)
    return r


def run_contingency_set(net, outage_list, voll=10000.0, redispatch_frac=None, rating_scale=None, label=""):
    """AC screening + DC load-shed for a list of outage sets. Returns a DataFrame."""
    dc = DCLoadShed(net, voll=voll, redispatch_frac=redispatch_frac, rating_scale=rating_scale)
    rows = []
    for outages in outage_list:
        r = ac_contingency(net, outages)
        s = dc.solve(outages)
        r.update(outage=";".join(f"{k[0].upper()}{i}" for k, i in outages), k=len(outages),
                 dc_ok=s["ok"], load_shed_mw=s["shed_mw"], n_islands=s["n_islands"])
        r["violation"] = (not r["converged"]) or r.get("n_overloads", 0) > 0 or r.get("n_voltage_violations", 0) > 0
        rows.append(r)
    df = pd.DataFrame(rows)
    df.insert(0, "case", label)
    return df

In [ ]:
WIND_CASE = f"fixed {FIXED_WIND} m/s"
pick = summ[(summ["wind_case"] == WIND_CASE) & (summ["Ts_max"] == COND["Ts_max"])].set_index("scenario")
RATIOS = {s: {"p50": float(pick.loc[s, "ratio_p50"]), "p10": float(pick.loc[s, "ratio_p10"])} for s in SCENARIOS}
base_ta = dc[dc["scenario"] == "historical"].set_index("model")["Ta"]
DT = {s: float((dc[dc["scenario"] == s].set_index("model")["Ta"] - base_ta).median()) for s in SCENARIOS}
with open(os.path.join(OUT, "derating_ratios.json"), "w") as f:
    json.dump({"site": SITE, "conductor": COND["name"], "Ts_max": COND["Ts_max"], "wind_case": WIND_CASE,
               "baseline": BASELINE, "future": FUTURE, "ratios": RATIOS, "delta_Ta_median": DT,
               "peak_factors": PF}, f, indent=2)
print("Derating ratios:", json.dumps(RATIOS, indent=1))
print("Median rise in design Tmax (°C):", {k: round(v, 2) for k, v in DT.items()})

In [ ]:
LOAD_SENS_PCT_PER_C = 2.0     # % peak-demand increase per °C rise in design Tmax (assumption)
HEAT_EVENT_H = 6              # hours per hot afternoon during which the derated limit binds
VOLL, REDISPATCH_FRAC = 10000.0, None    # keep identical to notebook 01

net_base = pp.from_json(require(os.path.join(RES["01_contingency"], "network_prepared.json"), "01"))

def heat_network(ratio, dT_load=0.0):
    n = copy.deepcopy(net_base)
    n.line["max_i_ka"] *= ratio
    if dT_load:
        f = 1 + LOAD_SENS_PCT_PER_C / 100 * dT_load
        n.load["p_mw"] *= f; n.load["q_mvar"] *= f
        n.gen["p_mw"] = np.minimum(n.gen.p_mw * f, n.gen.max_p_mw)    # generators follow the load
    return n

CASES = {"baseline": (1.0, 0.0)}
for s in SCENARIOS:
    CASES[f"{s} median"] = (RATIOS[s]["p50"], 0.0)
    CASES[f"{s} hot model (p10)"] = (RATIOS[s]["p10"], 0.0)
    CASES[f"{s} median + demand"] = (RATIOS[s]["p50"], DT[s])

n1_list = [[("line", i)] for i in net_base.line.index] + [[("trafo", i)] for i in net_base.trafo.index]
HEAT, NETS, BASE_SHED = {}, {}, {}
t0 = time.time()
for label, (ratio, dT_load) in CASES.items():
    n = heat_network(ratio, dT_load); pp.runpp(n)
    NETS[label] = n
    BASE_SHED[label] = DCLoadShed(n, VOLL, REDISPATCH_FRAC).solve([])
    HEAT[label] = run_contingency_set(n, n1_list, VOLL, REDISPATCH_FRAC, label=label)
    HEAT[label]["base_overloads"] = int((pd.concat([n.res_line.loading_percent, n.res_trafo.loading_percent]) > 100).sum())
print(f"{len(CASES)} cases x {len(n1_list)} contingencies in {time.time() - t0:.0f}s")
heat_n1 = pd.concat(HEAT.values(), ignore_index=True)
heat_n1.to_csv(os.path.join(OUT, "heat_n1_results.csv"), index=False)

In [ ]:
b = HEAT["baseline"].set_index("outage")
rows = []
for label, d in HEAT.items():
    d = d.set_index("outage")
    ratio, dT_load = CASES[label]
    rows.append({
        "case": label, "line rating ratio": ratio,
        "demand increase (%)": LOAD_SENS_PCT_PER_C * dT_load,
        "base-case overloads": d["base_overloads"].iloc[0],
        "base-case load shed (MW)": BASE_SHED[label]["shed_mw"],
        "N-1 with any violation": int(d.violation.sum()),
        "N-1 with thermal overload": int((d.n_overloads > 0).sum()),
        "new violations vs baseline": int((d.violation & ~b.violation.reindex(d.index)).sum()),
        "N-1 with load at risk": int((d.load_shed_mw > 0.1).sum()),
        "max N-1 load at risk (MW)": d.load_shed_mw.max(),
        "total overload above 100% (sum, %)": d.overload_excess_pct.sum(),
    })
heat_summary = pd.DataFrame(rows).set_index("case")
heat_summary.to_csv(os.path.join(OUT, "heat_n1_summary.csv"))
heat_summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
hs = heat_summary
c = ["#7f7f7f"] + ["#e08e0b" if "245" in k else "#c0392b" for k in hs.index[1:]]
ax[0].barh(hs.index[::-1], hs["N-1 with any violation"][::-1], color=c[::-1])
ax[0].set_xlabel("N-1 contingencies with a violation (of %d)" % len(n1_list))
ax[1].barh(hs.index[::-1], hs["total overload above 100% (sum, %)"][::-1], color=c[::-1])
ax[1].set_xlabel("Total post-contingency overload above 100% (sum over cases, %)")
ax[1].set_yticklabels([])
fig.suptitle(f"{SITE}: N-1 performance of {net_base.bus.shape[0]}-bus test network under 2041–2070 heat", fontsize=10)
savefig("fig_heat_n1.png")

In [ ]:
# Which contingencies become violations under future heat (SSP5-8.5, median model)?
key = "ssp585 median"
d = HEAT[key].set_index("outage")
newv = d[d.violation & ~b.violation.reindex(d.index)]
newv[["max_loading_pct", "n_overloads", "overloaded", "load_shed_mw"]].sort_values("max_loading_pct", ascending=False).head(15)

### 7.1 Export heat scenarios for the cascade analysis (notebook 04)
For each future case, every contingency (and the intact network) that forces load shedding becomes a scenario lasting `HEAT_EVENT_H` hours: an outage that happens during the hot afternoon window, when the derated limits bind. `heat_extra_mw` is the part of the shedding caused by heat, i.e. the extra shed beyond the same contingency at baseline ratings. If it is zero, corrective re-dispatch absorbed the derating; repeat with `REDISPATCH_FRAC = 0.2` to test a more constrained operator.

In [ ]:
meta_rows, ts = [], []
for label, n in NETS.items():
    if label == "baseline":
        continue
    slug = label.replace(" + ", "_plus_").replace(" ", "_").replace("(", "").replace(")", "")
    dcs = DCLoadShed(n, VOLL, REDISPATCH_FRAC)
    base_dc = DCLoadShed(NETS["baseline"], VOLL, REDISPATCH_FRAC)
    cand = [("base", [])] + [(o, [("line" if o[0] == "L" else "trafo", int(o[1:]))])
                             for o in HEAT[label].query("load_shed_mw > 0.1").outage]
    for name, outs in cand:
        r = dcs.solve(outs)
        if not r["ok"] or r["shed_mw"] <= 0.1:
            continue
        sid = f"heat_{slug}_{name}"
        s = r["shed_bus"][r["shed_bus"] > 1e-3]
        r0 = base_dc.solve(outs)
        meta_rows.append(dict(scenario_id=sid, family=f"heat: {label}", outages=name, n_out=len(outs),
                              peak_shed_mw=r["shed_mw"], heat_extra_mw=r["shed_mw"] - (r0["shed_mw"] if r0["ok"] else 0), duration_h=HEAT_EVENT_H, unserved_mwh=r["shed_mw"] * HEAT_EVENT_H))
        for h in range(HEAT_EVENT_H):
            ts.append(pd.DataFrame({"scenario_id": sid, "hour": h, "bus": s.index, "shed_mw": s.values}))
hmeta = pd.DataFrame(meta_rows, columns=["scenario_id", "family", "outages", "n_out", "peak_shed_mw", "heat_extra_mw", "duration_h", "unserved_mwh"])
hmeta.to_csv(os.path.join(OUT, "heat_scenarios_meta.csv"), index=False)
hts = pd.concat(ts, ignore_index=True) if ts else pd.DataFrame(columns=["scenario_id", "hour", "bus", "shed_mw"])
from_load = bus_loads(net_base)
hts["load_mw"] = hts.bus.map(from_load)
hts.to_csv(os.path.join(OUT, "heat_bus_shed_timeseries.csv"), index=False)
print(f"{len(hmeta)} heat scenarios exported")
hmeta.groupby("family").agg(scenarios=("scenario_id", "size"), max_shed_mw=("peak_shed_mw", "max"), heat_extra_mw=("heat_extra_mw", "sum"))

## 8. Notes for the technical note
- **Data:** NEX-GDDP-CMIP6 (Thrasher et al., 2022): 1985–2014 vs 2041–2070, SSP2-4.5 and SSP5-8.5, March–June, every model with `tasmax`, `sfcWind` and `rsds`. NASA POWER hourly data (2005–2024) give the diurnal factors.
- **Method:** IEEE 738-2012 steady-state rating, validated against the standard's Annex example. Ratios are computed per model and summarised as the ensemble median and 10th–90th percentiles, then applied to the N-1 analysis of notebook 01.
- **Headline result to report:** the number of N-1 contingencies with violations at baseline vs 2050 (median and hot model), and with demand growth.
- **Limitations:**
  - single 0.25° grid cell; the test network is not geographically located
  - the daily-to-hourly conversion assumes today's hot-day shape persists
  - climate-data radiation is used as the flux on the conductor
  - downscaled wind is uncertain, hence the fixed-wind headline case
  - approximate conductor resistances; one conductor type is applied to every line
  - steady-state (not transient) rating; sag clearances not assessed
  - assumed demand sensitivity

## 9. Fully open alternative to Earth Engine
The same NEX-GDDP-CMIP6 files are on the AWS Open Data registry (`s3://nex-gddp-cmip6`) and can be read anonymously with `xarray` + `s3fs` (+ `h5netcdf`); only Part 2 would change.

**Next:** notebook 04 (cascading impacts on hospitals and water).